# FineWeb Cleaning Workflow

**Purpose**: Apply targeted cleaning to FineWeb parquet slices before fine-tuning.

**Scope (Phase 1 — Pre-Easter)**:
1. **Intra-document line deduplication** — remove repeated lines within each document (nav bars, footers, repeated headers)
2. **Boilerplate stripping** — remove cookie banners, privacy notices, navigation fragments
3. **Cross-file URL deduplication** — ensure no URL appears more than once across all slices

**Why these three?** Our quality assessment found that **57.7% of sampled rows** had high character repetition. Most of this comes from repeated template lines within documents, not from truly duplicate documents. Boilerplate and URL duplicates are secondary but easy wins.

**Design**: Process one row group at a time (~1000 rows) to keep RAM constant at ~100MB regardless of file size.

## 1. Setup

In [ ]:
import os
import re
import time
from pathlib import Path
from collections import Counter

import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

# --- Auto-detect data directory ---
_candidates = [Path('/data/fineweb'), Path('data/fineweb'), Path('.')]
DATA_DIR = None
for d in _candidates:
    try:
        if d.exists() and list(d.glob('*.parquet')):
            DATA_DIR = d
            break
    except Exception:
        pass

if DATA_DIR is None:
    raise FileNotFoundError('No parquet files found.')

ALL_FILES = sorted(DATA_DIR.glob('*.parquet'))
OUTPUT_DIR = Path('cleaned')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Output directory: {OUTPUT_DIR.resolve()}')
print(f'Found {len(ALL_FILES)} parquet file(s)')

## 2. Cleaning Functions

In [ ]:
def dedup_lines_in_doc(text, discard_threshold=0.5, dedup_threshold=0.2):
    """Remove duplicate lines within a single document.
    - discard_threshold: if >50% lines are dups, discard entire doc
    - dedup_threshold: if >20% lines are dups, remove dup lines (keep first)
    """
    if not text or not text.strip():
        return None
    lines = text.split('\n')
    if len(lines) <= 1:
        return text
    line_counts = Counter(line.strip() for line in lines if line.strip())
    total_lines = sum(line_counts.values())
    dup_lines = sum(c - 1 for c in line_counts.values() if c > 1)
    if total_lines == 0:
        return None
    dup_ratio = dup_lines / total_lines
    if dup_ratio > discard_threshold:
        return None
    if dup_ratio > dedup_threshold:
        seen = set()
        deduped = []
        for line in lines:
            stripped = line.strip()
            if not stripped:
                deduped.append(line)
                continue
            if stripped not in seen:
                seen.add(stripped)
                deduped.append(line)
        return '\n'.join(deduped)
    return text


BOILERPLATE_LINE_PATTERNS = re.compile(
    r'^\s*('
    r'(accept|manage|reject)\s*(all)?\s*cookies?'
    r'|we use cookies|this (site|website) uses cookies'
    r'|cookie (policy|settings|preferences)|privacy policy'
    r'|terms (of|and) (service|use|conditions)|all rights reserved'
    r'|subscribe to (our )?newsletter|sign up for (our )?newsletter'
    r'|click here to|share (on|this|via)\s*(facebook|twitter|linkedin|x|email)?'
    r'|follow us (on)?|\u00a9\s*\d{4}|powered by\s+\w+'
    r'|skip to (main )?content|toggle (navigation|menu)'
    r'|search(\\.\\.\\.)?$|log\s*in$|sign\s*(in|up)$|menu$'
    r')\s*$',
    re.IGNORECASE
)


def strip_boilerplate(text):
    """Remove lines that are entirely boilerplate."""
    if not text:
        return text
    lines = text.split('\n')
    cleaned = [l for l in lines if not BOILERPLATE_LINE_PATTERNS.match(l)]
    result = '\n'.join(cleaned).strip()
    return None if len(result) < 50 else result


class URLDeduplicator:
    """Track seen URLs across all files for global deduplication."""
    def __init__(self):
        self.seen = set()
        self.dup_count = 0
    def is_duplicate(self, url):
        if url in self.seen:
            self.dup_count += 1
            return True
        self.seen.add(url)
        return False
    def dedup_df(self, df):
        if 'url' not in df.columns:
            return df
        mask = ~df['url'].apply(lambda u: self.is_duplicate(u))
        return df[mask]
    def stats(self):
        return {'unique_urls': len(self.seen), 'duplicates_removed': self.dup_count}


print('Cleaning functions loaded.')

## 3. Pipeline

In [ ]:
PIPELINE_CONFIG = {
    'line_dedup_discard': 0.5,
    'line_dedup_threshold': 0.2,
    'min_tokens': 50,
    'min_text_length': 100,
    'min_lang_score': 0.65,
}


def clean_text(text, config):
    """Apply text-level cleaning. Returns cleaned text or None."""
    text = dedup_lines_in_doc(text,
        discard_threshold=config['line_dedup_discard'],
        dedup_threshold=config['line_dedup_threshold'])
    if text is None:
        return None
    text = strip_boilerplate(text)
    return text


def clean_row_group(df, url_dedup, config):
    """Clean one row group. Returns (cleaned_df, stats_counter)."""
    stats = Counter()
    n_original = len(df)
    cleaned_texts = []
    keep_mask = []
    for text in df['text']:
        result = clean_text(str(text) if pd.notna(text) else '', config)
        if result is None:
            keep_mask.append(False)
            cleaned_texts.append('')
            stats['text_discarded'] += 1
        else:
            keep_mask.append(True)
            cleaned_texts.append(result)
    df = df[keep_mask].copy()
    df['text'] = [t for t, k in zip(cleaned_texts, keep_mask) if k]
    if 'token_count' in df.columns:
        mask = df['token_count'] >= config['min_tokens']
        stats['too_few_tokens'] = int((~mask).sum())
        df = df[mask]
    if 'language_score' in df.columns:
        mask = df['language_score'] >= config['min_lang_score']
        stats['low_lang_score'] = int((~mask).sum())
        df = df[mask]
    mask = df['text'].str.len() >= config['min_text_length']
    stats['too_short_after_clean'] = int((~mask).sum())
    df = df[mask]
    before_url = len(df)
    df = url_dedup.dedup_df(df)
    stats['url_duplicate'] = before_url - len(df)
    stats['kept'] = len(df)
    stats['removed'] = n_original - len(df)
    stats['original'] = n_original
    return df, stats


def clean_single_file(filepath, url_dedup, config, output_dir, log_fn=print):
    """Clean one parquet file end-to-end. Returns stats dict."""
    pf = pq.ParquetFile(filepath)
    file_stats = Counter()
    clean_tables = []
    t0 = time.time()

    for gi in range(pf.num_row_groups):
        df = pf.read_row_group(gi).to_pandas()
        cleaned, stats = clean_row_group(df, url_dedup, config)
        file_stats += stats
        if len(cleaned) > 0:
            clean_tables.append(pa.Table.from_pandas(cleaned, preserve_index=False))
        del df, cleaned
        if (gi + 1) % 20 == 0:
            log_fn(f'  ... {gi+1}/{pf.num_row_groups} groups')

    output_path = output_dir / filepath.name
    out_size = 0
    if clean_tables:
        combined = pa.concat_tables(clean_tables)
        pq.write_table(combined, output_path)
        out_size = output_path.stat().st_size / (1024**2)

    elapsed = time.time() - t0
    in_size = filepath.stat().st_size / (1024**2)

    result = {
        'file': filepath.name,
        'original_rows': file_stats['original'],
        'kept_rows': file_stats['kept'],
        'removed_rows': file_stats['removed'],
        'text_discarded': file_stats['text_discarded'],
        'too_few_tokens': file_stats['too_few_tokens'],
        'low_lang_score': file_stats['low_lang_score'],
        'too_short_after_clean': file_stats['too_short_after_clean'],
        'url_duplicate': file_stats['url_duplicate'],
        'keep_rate': round(file_stats['kept'] / max(file_stats['original'], 1) * 100, 1),
        'input_mb': round(in_size, 1),
        'output_mb': round(out_size, 1),
        'time_sec': round(elapsed, 1),
    }
    return result


print('Pipeline ready.')

## 4. Interactive File Selector

Select which slices to clean. Already-cleaned files are shown with a checkmark and cannot be re-selected.

In [ ]:
# ============================================================
# Interactive cleaning UI
# ============================================================

url_dedup = URLDeduplicator()
all_cleaning_stats = []

# --- UI components ---
status_out = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px 0'))
file_box = widgets.VBox()
button_bar = widgets.HBox()
progress_out = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px 0'))

checkboxes = {}  # filepath -> Checkbox widget


def get_file_info(f):
    """Get metadata for a parquet file."""
    pf = pq.ParquetFile(f)
    size_mb = f.stat().st_size / (1024**2)
    est_rows = sum(pf.metadata.row_group(j).num_rows for j in range(pf.num_row_groups))
    return size_mb, est_rows, pf.num_row_groups


def is_cleaned(f):
    """Check if a file has already been cleaned."""
    return (OUTPUT_DIR / f.name).exists()


def build_ui():
    """Build/rebuild the file selector UI."""
    checkboxes.clear()
    file_widgets = []
    n_cleaned = 0
    n_uncleaned = 0

    for f in ALL_FILES:
        size_mb, est_rows, n_groups = get_file_info(f)

        if is_cleaned(f):
            n_cleaned += 1
            out_size = (OUTPUT_DIR / f.name).stat().st_size / (1024**2)
            label = widgets.HTML(
                value=f'<span style="color:#4caf50; margin-left:26px;">'
                      f'\u2705 {f.name} &nbsp; ({size_mb:.0f} \u2192 {out_size:.0f} MB, ~{est_rows:,} rows) &mdash; cleaned</span>',
                layout=widgets.Layout(margin='2px 0')
            )
            file_widgets.append(label)
        else:
            n_uncleaned += 1
            cb = widgets.Checkbox(
                value=False,
                description=f'{f.name}   ({size_mb:.0f} MB, ~{est_rows:,} rows, {n_groups} groups)',
                indent=False,
                layout=widgets.Layout(width='100%', margin='2px 0'),
                style={'description_width': 'initial'},
            )
            checkboxes[f] = cb
            file_widgets.append(cb)

    file_box.children = file_widgets

    with status_out:
        clear_output(wait=True)
        if n_cleaned == len(ALL_FILES):
            print(f'All {len(ALL_FILES)} files have been cleaned!')
        else:
            print(f'{n_cleaned} cleaned  |  {n_uncleaned} remaining  |  {len(ALL_FILES)} total')

    # Disable buttons if nothing to clean
    clean_btn.disabled = (n_uncleaned == 0)
    select_all_btn.disabled = (n_uncleaned == 0)


def on_select_all(b):
    """Toggle all uncleaned checkboxes."""
    all_checked = all(cb.value for cb in checkboxes.values())
    for cb in checkboxes.values():
        cb.value = not all_checked


def on_clean_click(b):
    """Run cleaning on selected files."""
    selected = [f for f, cb in checkboxes.items() if cb.value]
    if not selected:
        with progress_out:
            clear_output(wait=True)
            print('No files selected. Check the boxes above first.')
        return

    clean_btn.disabled = True
    select_all_btn.disabled = True
    clean_btn.description = 'Cleaning...'

    with progress_out:
        clear_output(wait=True)
        print(f'Starting cleaning of {len(selected)} file(s)...\n')

        for fi, filepath in enumerate(selected):
            print(f'{"="*70}')
            print(f'[{fi+1}/{len(selected)}] {filepath.name}')

            result = clean_single_file(
                filepath, url_dedup, PIPELINE_CONFIG, OUTPUT_DIR, log_fn=print
            )
            all_cleaning_stats.append(result)

            print(f'  Original: {result["original_rows"]:,} rows ({result["input_mb"]} MB)')
            print(f'  Kept:     {result["kept_rows"]:,} rows ({result["output_mb"]} MB)  [{result["keep_rate"]}%]')
            print(f'  Removed:  {result["removed_rows"]:,} rows'
                  f'  (text={result["text_discarded"]}, tokens={result["too_few_tokens"]},'
                  f' lang={result["low_lang_score"]}, short={result["too_short_after_clean"]},'
                  f' url_dup={result["url_duplicate"]})')
            print(f'  Time: {result["time_sec"]}s\n')

        print(f'{"="*70}')
        print(f'Done! URL dedup stats: {url_dedup.stats()}')

    clean_btn.description = 'Clean Selected'
    build_ui()  # refresh to show newly cleaned files


# --- Create buttons ---
select_all_btn = widgets.Button(
    description='Select All / None',
    button_style='info',
    layout=widgets.Layout(width='180px', height='36px'),
)
clean_btn = widgets.Button(
    description='Clean Selected',
    button_style='success',
    layout=widgets.Layout(width='180px', height='36px'),
)
select_all_btn.on_click(on_select_all)
clean_btn.on_click(on_clean_click)
button_bar.children = [select_all_btn, clean_btn]

# --- Build and display ---
build_ui()
display(status_out, file_box, button_bar, progress_out)

## 5. Export Cleaning Report (CSV)

Run this cell after cleaning to generate the summary CSV.

In [ ]:
if all_cleaning_stats:
    REPORT_DIR = Path('outputs')
    REPORT_DIR.mkdir(exist_ok=True)

    report_df = pd.DataFrame(all_cleaning_stats)
    totals = report_df.select_dtypes(include='number').sum()
    totals['file'] = 'TOTAL'
    totals['keep_rate'] = round(totals['kept_rows'] / totals['original_rows'] * 100, 1)
    report_df = pd.concat([report_df, pd.DataFrame([totals])], ignore_index=True)

    report_df.to_csv(REPORT_DIR / 'cleaning_report.csv', index=False)
    print('cleaning_report.csv saved')
    print(report_df.to_string(index=False))

    print(f'\n--- Summary ---')
    t = totals
    print(f'Total original : {int(t["original_rows"]):>10,} rows')
    print(f'Total kept     : {int(t["kept_rows"]):>10,} rows')
    print(f'Total removed  : {int(t["removed_rows"]):>10,} rows ({100-t["keep_rate"]:.1f}%)')
    print(f'\nURL dedup: {url_dedup.stats()}')
else:
    print('No cleaning has been performed yet. Use the selector above first.')

## 6. Verify Cleaned Output

Spot-check a sample from the first cleaned file.

In [ ]:
# Find the first cleaned file
cleaned_files = sorted(OUTPUT_DIR.glob('*.parquet'))
if cleaned_files:
    cf = cleaned_files[0]
    orig_f = DATA_DIR / cf.name

    orig_pf = pq.ParquetFile(orig_f)
    clean_pf = pq.ParquetFile(cf)

    orig_df = orig_pf.read_row_group(0).to_pandas()
    clean_df = clean_pf.read_row_group(0).to_pandas()

    print(f'File: {cf.name}')
    print(f'Original row group 0: {len(orig_df)} rows')
    print(f'Cleaned  row group 0: {len(clean_df)} rows')

    if len(clean_df) > 0:
        row = clean_df.iloc[0]
        print(f'\n--- Sample cleaned record ---')
        print(f'URL: {row["url"]}')
        print(f'Tokens: {row["token_count"]}  |  Lang score: {row["language_score"]}')
        text = row['text']
        print(f'Text ({len(text)} chars):')
        print(text[:800] + ('\n... [truncated]' if len(text) > 800 else ''))
else:
    print('No cleaned files found yet. Run the cleaner above first.')

## Next Steps (Post-Easter)

**Phase 2 — Content quality scoring:**
- Character n-gram diversity score
- Sentence structure analysis (avg sentence length, punctuation density)
- Uppercase ratio filtering
- Alphabetic content ratio
- Optional: perplexity scoring with a small language model

**Phase 3 — Near-duplicate detection:**
- MinHash / LSH for fuzzy deduplication across documents
- Jaccard similarity threshold tuning

**Phase 4 — Fine-tuning pipeline integration:**
- Output cleaned text in training-ready format
- Connect to tokenizer and data loader